In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import quad
from scipy.optimize import curve_fit

"""
Herramientas Hidrogeológicas
Pruebas de bombeo en acuíferos semiconfinados (Método de Hantush-Walton)

Descripción: Ajuste inverso automatizado para estimar la Transmisividad (T), 
Coeficiente de Almacenamiento (S) y factor de goteo (r/B) mediante integración numérica.

Autor: Carlos Javier Pérez Pérez
"""

In [ ]:
# 1. Datos de campo y parámetros iniciales
df = pd.read_excel('DatosCampoHantush.xlsx')

caudal_lps = 15.0       
radio_obs_m = 20.0
espesor_acuitardo = 5.0  # b' (metros)

# Conversión a unidades estándar (m³/día y días)
caudal_m3d = caudal_lps * 86.4 
df['tiempo_dias'] = df['tiempo_min'] / 1440.0

In [ ]:
# 2. Definición del modelo analítico de Hantush
def integral_hantush(u, r_B):
    """Calcula la función de pozo W(u, r/B) numéricamente."""
    res = np.zeros_like(u)
    for i, u_val in enumerate(u):
        def integrando(y):
            return (1/y) * np.exp(-y - (r_B**2)/(4*y))
        res[i], _ = quad(integrando, u_val, np.inf)
    return res

def modelo_hantush(t_dias, T, S, r_B):
    """Modelo de abatimiento s(t) para regresión no lineal."""
    u = (radio_obs_m**2 * S) / (4 * T * t_dias)
    W_u = integral_hantush(u, r_B)
    descenso = (caudal_m3d / (4 * np.pi * T)) * W_u
    return descenso

In [ ]:
# 3. Ajuste Automático (Curve Fitting)
estimacion_inicial = [50.0, 0.001, 0.2] # [T inicial, S inicial, r/B inicial]
limites = ([1.0, 1e-6, 0.01], [1000.0, 0.1, 5.0])

# El algoritmo encuentra la curva teórica perfecta
parametros_opt, covarianza = curve_fit(
    modelo_hantush, 
    df['tiempo_dias'], 
    df['descenso_m'], 
    p0=estimacion_inicial,
    bounds=limites
)

T_calc, S_calc, r_B_calc = parametros_opt

# Propiedades del acuitardo
B_calc = radio_obs_m / r_B_calc
K_prima_calc = (T_calc * espesor_acuitardo) / (B_calc**2)

In [ ]:
# 4. Reporte en consola
print("Reporte Hidrogeológico: Método de Hantush-Walton")
print("-" * 50)
print(f"Transmisividad (T)             : {T_calc:.1f} m²/día")
print(f"Almacenamiento (S)             : {S_calc:.2e}")
print(f"Factor de ajuste (r/B)         : {r_B_calc:.3f}")
print(f"Factor de goteo (B)            : {B_calc:.1f} m")
print(f"Conductiv. del acuitardo (K')  : {K_prima_calc:.4f} m/día\n")

In [ ]:
# 5. Configuración y trazado del gráfico de diagnóstico
plt.rcParams.update({'font.family': 'serif', 'font.size': 11})
fig, ax = plt.subplots(figsize=(10, 6))

# Generar la curva teórica óptima encontrada
t_teorico = np.logspace(np.log10(df['tiempo_dias'].min()), np.log10(df['tiempo_dias'].max()), 100)
s_teorico = modelo_hantush(t_teorico, T_calc, S_calc, r_B_calc)

# Trazado
ax.semilogx(df['tiempo_min'], df['descenso_m'], 's', color='#b30000', markersize=8, label='Medidas De Campo')
ax.semilogx(t_teorico * 1440.0, s_teorico, color='#2c3e50', linewidth=2, label=f'Curva Teórica Óptima (r/B = {r_B_calc:.3f})')

ax.grid(True, which="both", ls="--", alpha=0.5, color='gray')
ax.set_xlabel('Tiempo En Escala Logarítmica (minutos)', fontweight='bold')
ax.set_ylabel('Descenso $s$ (metros)', fontweight='bold')
ax.set_title('Análisis De Prueba De Bombeo: Método De Hantush-Walton', pad=15, fontweight='bold')

# Caja de resultados
texto_resultados = (
    f"Parámetros Estimados:\\n"
    f"T = {T_calc:.1f} m²/d\\n"
    f"S = {S_calc:.1e}\\n"
    f"r/B = {r_B_calc:.3f}\\n"
    f"K' = {K_prima_calc:.4f} m/d"
)
props_caja = dict(boxstyle='square,pad=0.6', facecolor='#f9f9f9', edgecolor='black', alpha=0.9)
ax.text(0.04, 0.96, texto_resultados, transform=ax.transAxes, fontsize=10, verticalalignment='top', bbox=props_caja)

ax.legend(loc='lower right', framealpha=1.0, edgecolor='black')
plt.tight_layout()
plt.show()